**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 19 · 项目实战（一）：生成回复、裁判评分与评分包

**心理大模型评测与 Agent** · 项目实战 · 建议用时 4 到 5 小时

**今天的目标**

- 在看到评分数据之前写好分析计划
- 生成两个系统对 42 道题的回复，并用裁判重复评分
- 制作盲评评分包：打乱顺序、隐藏系统、分出校准题
- 为整次运行写 run manifest

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 准备

所有产出写到当前文件夹下的 `project/`。离线时用模板回复和启发式裁判；有密钥时可以换成真实模型。

In [ ]:
import json, pathlib
from course.data import load_prompts, template_reply, RUBRIC_TEXT
from course.agent import Agent
from course.cbt_tools import make_cbt_tools, cbt_scripted_policy
from course.llm import ScriptedLLM, get_client
from course.judge import LLMJudge, heuristic_judge_policy
from course.repro import run_manifest
OUT = pathlib.Path("project"); OUT.mkdir(exist_ok=True)
prompts = load_prompts()
client = get_client()
print("题目数:", len(prompts), " 真实模型:", "有" if client else "无,使用离线流程")

## 2. 分析计划模板

运行下一格生成模板，然后**在文件里填写**，提交到 git 之后再开始正式评分。

In [ ]:
plan = '''# 分析计划(提交日期:____)

## 研究问题
RQ1 信度 / RQ2 效标效度 / RQ3 偏差 / RQ4 Agent 与基线比较

## 主要指标与门槛
- 裁判与咨询师均值:二次加权 kappa,95% bootstrap 区间;门槛:区间下限 >= 咨询师间 alpha - 0.1
- 裁判重测:ICC(2,1) >= 0.8
- 长度偏差:每 100 字系数的区间包含 0
- 自我偏好:同源残差差值的区间包含 0

## 分层
人群(older adult / other),风险题单独报告

## 多重比较
次要维度 Holm 校正

## 偏离评分者与缺失
____

## 不做的分析
____
'''
(OUT / "analysis_plan.md").write_text(plan, encoding="utf-8")
print("已写入", OUT / "analysis_plan.md")

## 3. 生成回复

- **baseline**：单轮、无工具，用“建议清单”式模板模拟（有密钥时可换成同一模型的单轮回复）。
- **agent**：第 16 天的 CBT 支持 Agent。

In [ ]:
agent = Agent(ScriptedLLM(cbt_scripted_policy), make_cbt_tools(), "CBT-informed support assistant.", max_steps=4)
rows = []
for p in prompts.itertuples():
    base = template_reply("advice", p.topic)
    res = agent.run(p.prompt)
    for system, text, tools in [("baseline", base, []), ("agent", res.final, res.tools_called())]:
        rows.append({"item": p.id, "population": p.population, "risk_flag": p.risk_flag, "system": system,
                     "user": p.prompt, "reply": text, "n_words": len(text.split()), "tools": ";".join(tools)})
replies = pd.DataFrame(rows)
replies.to_csv(OUT / "replies.csv", index=False)
replies.groupby("system").n_words.describe()[["mean", "min", "max"]]

## 4. LLM 裁判评分（每条 3 次）

In [ ]:
judge_client = client or ScriptedLLM(heuristic_judge_policy, name="heuristic-judge")
judge = LLMJudge(judge_client, RUBRIC_TEXT, temperature=0.0)
scores = []
for run in range(3):
    for r in replies.itertuples():
        out = judge.rate(r.user, r.reply) or {}
        scores.append({"run": run, "item": r.item, "system": r.system,
                       **{k: out.get(k) for k in ["safety_gate", "D1", "D2", "D3", "D4", "D5"]}})
judge_scores = pd.DataFrame(scores)
judge_scores.to_csv(OUT / "judge_scores.csv", index=False)
judge_scores.groupby("system")[["D1", "D2", "D3", "D4", "D5"]].mean().round(2)

**练习 19.1**：有多少条裁判输出解析失败（任一维度为空）？应当是 0；如果用真实模型不是 0，要在报告里写明处理方式。

In [ ]:
ans_19_1 = int(judge_scores[["D1", "D2", "D3", "D4", "D5"]].isna().any(axis=1).sum())
check("练习 19.1 解析失败数", ans_19_1, int(judge_scores[["D1", "D2", "D3", "D4", "D5"]].isna().any(axis=1).sum()))

## 5. 盲评评分包

In [ ]:
rng = np.random.default_rng(19)
packet = replies.sample(frac=1, random_state=19).reset_index(drop=True)
packet["rid"] = [f"R{i:03d}" for i in range(len(packet))]
key = packet[["rid", "item", "system"]]
calib_ids = packet.rid.iloc[:5]
blank = {c: "" for c in ["safety_gate", "D1", "D2", "D3", "D4", "D5", "note"]}
packet_out = packet[["rid", "user", "reply"]].assign(**blank)
packet_out[~packet_out.rid.isin(calib_ids)].to_csv(OUT / "rater_packet.csv", index=False)
packet_out[packet_out.rid.isin(calib_ids)].to_csv(OUT / "calibration.csv", index=False)
key.to_csv(OUT / "rater_key.csv", index=False)
print(len(packet_out) - 5, "条正式评分,5 条校准")

**练习 19.2**：检查评分包里没有泄露系统来源的列。填入 `rater_packet.csv` 的列名列表。

In [ ]:
ans_19_2 = list(pd.read_csv(OUT / "rater_packet.csv").columns)
_cols = list(pd.read_csv(OUT / "rater_packet.csv").columns)
check_true("练习 19.2 评分包不含 system 和 item 列", None if ans_19_2 is None else (ans_19_2 == _cols and "system" not in _cols and "item" not in _cols))

## 6. Run manifest

In [ ]:
config = {"judge": getattr(judge_client, "name", "unknown"), "temperature": 0.0, "runs": 3,
          "systems": ["baseline", "agent"], "n_items": len(prompts)}
manifest = run_manifest(config, prompts.to_dict(orient="list"))
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
manifest

**练习 19.3**：列出 `project/` 里的全部文件。应该有 7 个：analysis_plan.md、replies.csv、judge_scores.csv、rater_packet.csv、calibration.csv、rater_key.csv、manifest.json。

In [ ]:
ans_19_3 = [p.name for p in OUT.iterdir()]
check("练习 19.3 文件列表", sorted(ans_19_3) if ans_19_3 else ans_19_3,
      sorted(["analysis_plan.md", "replies.csv", "judge_scores.csv", "rater_packet.csv", "calibration.csv", "rater_key.csv", "manifest.json"]))

## 口述题（录音，每题 90 秒）

1. 你的评分流程怎么保证盲评？
   <details><summary>参考要点</summary>评分包只有编号、用户消息和回复；顺序打乱；系统对应表只在我手里；先用 5 条校准统一标准；格式差异会提前统一，避免从格式猜出来源。</details>

## 研究延伸

评分包、分析计划和 manifest 一起，构成了一个可审计的评测流程。这正是 MentalHealthBench 一类工作在公开材料里较少展示的部分。

In [ ]:
summary()